# IKG Table Lineage Dependency Builder

Use this notebook to explore dependency chains across profiles captured in `sandbox_prj_smart_insights.ikg_table_lineage_metadata_auto_refresh` and materialize ordered execution plans in both Excel and Greenplum.

In [ ]:
import datetime
import logging

from ikg_table_lineage_dependency_builder import (
    build_adjacency,
    build_db_config,
    build_lineage_rows,
    fetch_metadata_rows,
    fetch_profile_tables,
    rows_to_dataframe,
    write_temp_excel,
    refresh_temp_table,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
print("Dependency builder utilities loaded.")

In [ ]:
import getpass

insight_types_raw = input("Enter insight_type (comma separated): ").strip()
INSIGHT_TYPES = [part.strip() for part in insight_types_raw.split(",") if part.strip()]
if not INSIGHT_TYPES:
    raise ValueError("Please provide at least one insight_type.")

DB_PASSWORD = getpass.getpass("Enter Password for DB User: ")
DB_CONFIG = build_db_config(DB_PASSWORD)
print(f"Captured {len(INSIGHT_TYPES)} insight type(s).")

In [ ]:
import psycopg2

with psycopg2.connect(**DB_CONFIG) as conn:
    profile_tables = fetch_profile_tables(conn, INSIGHT_TYPES)
    if not profile_tables:
        raise RuntimeError("No profile tables found for the requested insight types.")
    metadata_rows = fetch_metadata_rows(conn)

print(f"Profile tables: {profile_tables}")
print(f"Metadata rows fetched: {len(metadata_rows)}")

In [ ]:
adjacency = build_adjacency(metadata_rows)
run_ts = datetime.datetime.utcnow()
lineage_rows = build_lineage_rows(profile_tables, adjacency, run_ts)
print(f"Generated {len(lineage_rows)} dependency rows.")
lineage_rows[:5]

In [ ]:
df = rows_to_dataframe(lineage_rows)
output_file = write_temp_excel(df, run_ts)
print(f"Excel saved to {output_file}")
df.head()

In [ ]:
with psycopg2.connect(**DB_CONFIG) as conn:
    refresh_temp_table(conn, lineage_rows)

print("Greenplum temp table refreshed.")